# Black-box Question Generation Pipeline

This notebook generates domain-adapted English questions using predefined templates and entity slots.
It is designed to be **non-iterative** (one-shot generation) and suitable for testing across different domain descriptions.

Notebook structure:
1. Step 1 — Basic setup (templates & domain weights)
2. Step 2 — Input database description
3. Step 3 — Allocate templates according to domain
4. Step 4 — Generate English questions (slot filling)
5. Step 5 — Export results

## 1. Step 1 — Basic setup (templates & domain weights)

In [49]:
import random
import json

# English template library (A-F categories)
TEMPLATES = {
    "A": [
        "What is [ENTITY]?",
        "What are the main features of [ENTITY]?",
        "Can you give an example of [ENTITY]?",
        "How does [ENTITY A] differ from [ENTITY B]?"
    ],
    "B": [
        "What are the key steps in [PROCESS]?",
        "How is [TASK] performed?",
        "Which methods are usually used for [TASK]?"
    ],
    "C": [
        "Why does [PHENOMENON] occur?",
        "What are the causes of [PHENOMENON]?",
        "What evidence supports [CLAIM]?"
    ],
    "D": [
        "How has [ENTITY] changed over time?",
        "What are the major historical milestones of [ENTITY]?",
        "What trends can be observed in [ENTITY]?"
    ],
    "E": [
        "What is the practical application of [ENTITY]?",
        "How does [ENTITY] relate to real-world problems?",
        "What impact might [ENTITY] have on society?"
    ],
    "F": [
        "What are the main controversies about [ENTITY] in academia?",
        "How do different viewpoints on [ENTITY] conflict?",
        "What gaps currently exist in research on [ENTITY]?"
    ]
}

# Domain weights mapping (A-F categories)
DOMAIN_WEIGHTS = {
    "General Knowledge":  {"A":0.30,"B":0.15,"C":0.15,"D":0.15,"E":0.15,"F":0.10},
    "Academic/Research":   {"A":0.15,"B":0.25,"C":0.15,"D":0.20,"E":0.15,"F":0.10},
    "Medical/Clinical":    {"A":0.15,"B":0.30,"C":0.25,"D":0.10,"E":0.15,"F":0.05},
    "Legal/Regulations":   {"A":0.10,"B":0.20,"C":0.30,"D":0.20,"E":0.15,"F":0.05},
    "News/Current Events": {"A":0.20,"B":0.15,"C":0.20,"D":0.15,"E":0.20,"F":0.10},
    "Social Media/Chat":   {"A":0.15,"B":0.15,"C":0.15,"D":0.10,"E":0.30,"F":0.15},
    "Technical Docs/FAQ":  {"A":0.25,"B":0.25,"C":0.30,"D":0.10,"E":0.05,"F":0.05},
    "Historical Archives": {"A":0.20,"B":0.20,"C":0.15,"D":0.25,"E":0.10,"F":0.10},
    "Finance":             {"A":0.25,"B":0.25,"C":0.15,"D":0.10,"E":0.15,"F":0.10} 
}

## 2. Step 2 — Input database description

In [50]:
database_desc = {
    "name": "chatdoctor",
    "type": "Medical/Clinical",  # 映射英文类别
    "intro": "Real conversations between patients and doctors from multiple medical dialogue websites, containing a wealth of authentic case records covering disease symptoms, diagnoses, and treatment recommendations."
}

# database_desc = {
#     "name": "fiqa",
#     "type": "Finance",  # 映射英文类别
#     "intro": "a financial sentiment analysis benchmark derived from real-world sources such as StockTwits posts and financial news headlines.it enables models to understand market sentiment and investor opinions in financial contexts."
# }

print("Database description:")
for k, v in database_desc.items():
    print(f"{k}: {v}")

Database description:
name: chatdoctor
type: Medical/Clinical
intro: Real conversations between patients and doctors from multiple medical dialogue websites, containing a wealth of authentic case records covering disease symptoms, diagnoses, and treatment recommendations.


## 3. Step 3 — Allocate templates according to domain

In [51]:
# Allocate templates according to domain

def allocate_templates(domain_type, total_questions=50):
    # 根据 domain_type 从 DOMAIN_WEIGHTS 获取比例，并计算每类模板数量
    if domain_type not in DOMAIN_WEIGHTS:
        raise ValueError(f"Domain type '{domain_type}' not found in DOMAIN_WEIGHTS")
    weights = DOMAIN_WEIGHTS[domain_type]
    allocation = {k: int(v * total_questions) for k, v in weights.items()}
    return allocation

# 分配模板
allocation = allocate_templates(database_desc["type"], total_questions=500)
print("Template allocation (number of questions per category):")
print(allocation)

Template allocation (number of questions per category):
{'A': 75, 'B': 150, 'C': 125, 'D': 50, 'E': 75, 'F': 25}


## 4. Step 4 — Generate English questions (slot filling)

In [54]:
### get entity pool
from openai import OpenAI

# 初始化 LLM（示例为 OpenAI API）
client = OpenAI(base_url="http://localhost:22999/v1", api_key="EMPTY")

def generate_entity_pool_llm(db_description, num_entities=30):
    prompt = f"""
    Given the following database description:
    \"\"\"{db_description['intro']}\"\"\"

    Please generate a list of {num_entities} relevant entities in English, without any extra explanation, prefix and suffix. Output as a JSON array.
    """

    response = client.chat.completions.create(
        model="./Models/Qwen3-32B",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
        top_p=0.8,
        max_tokens=2048
    )
    response

    return response

# 生成实体池
response = generate_entity_pool_llm(database_desc, num_entities=100)

ChatCompletion(id='chatcmpl-f5b723a207814efcba60a713395c1b42', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='\n\n```json\n[\n    "Hypertension",\n    "Diabetes",\n    "Asthma",\n    "Arthritis",\n    "Pneumonia",\n    "Migraine",\n    "Depression",\n    "Anxiety",\n    "Gastritis",\n    "Allergies",\n    "Cancer",\n    "Heartburn",\n    "Insomnia",\n    "Fever",\n    "Headache",\n    "Nausea",\n    "Fatigue",\n    "Dizziness",\n    "Cough",\n    "Shortness of Breath",\n    "Back Pain",\n    "Joint Pain",\n    "Abdominal Pain",\n    "Diarrhea",\n    "Constipation",\n    "Vomiting",\n    "Rash",\n    "Itching",\n    "Weight Loss",\n    "Weight Gain",\n    "Swelling",\n    "Chest Pain",\n    "Palpitations",\n    "High Cholesterol",\n    "High Blood Sugar",\n    "Low Blood Pressure",\n    "Kidney Stones",\n    "Urinary Tract Infection",\n    "Sinusitis",\n    "Bronchitis",\n    "Eczema",\n    "Psoriasis",\n    "Meningitis",\n    "Hepat

In [58]:
print(response.choices[0].message.content.replace("\n", ""))

```json[    "Hypertension",    "Diabetes",    "Asthma",    "Arthritis",    "Pneumonia",    "Migraine",    "Depression",    "Anxiety",    "Gastritis",    "Allergies",    "Cancer",    "Heartburn",    "Insomnia",    "Fever",    "Headache",    "Nausea",    "Fatigue",    "Dizziness",    "Cough",    "Shortness of Breath",    "Back Pain",    "Joint Pain",    "Abdominal Pain",    "Diarrhea",    "Constipation",    "Vomiting",    "Rash",    "Itching",    "Weight Loss",    "Weight Gain",    "Swelling",    "Chest Pain",    "Palpitations",    "High Cholesterol",    "High Blood Sugar",    "Low Blood Pressure",    "Kidney Stones",    "Urinary Tract Infection",    "Sinusitis",    "Bronchitis",    "Eczema",    "Psoriasis",    "Meningitis",    "Hepatitis",    "Tuberculosis",    "HIV",    "Diabetic Neuropathy",    "Osteoporosis",    "Thyroid Disorders",    "Epilepsy",    "Parkinson's Disease",    "Alzheimer's Disease",    "Multiple Sclerosis",    "Chronic Obstructive Pulmonary Disease",    "Gout",    "Ul

In [62]:
entity_pool = ['Itching',
 'Thyroid Disorders',
 'Diet Plan',
 'Chest Pain',
 'Eczema',
 'Postpartum Depression',
 'Weight Gain',
 'Diuretics',
 'Rehabilitation',
 'Chemotherapy',
 'Diarrhea',
 'Colonoscopy',
 'Chronic Cough',
 'Heartburn',
 'Breast Cancer',
 'Radiation Therapy',
 'Osteoarthritis',
 'Pain Management',
 'Anticonvulsants',
 'Celiac Disease',
 'Hypertension',
 'Lifestyle Modifications',
 'Low Blood Pressure',
 'Anti-inflammatory Drugs',
 'Meningitis',
 'Sleep Aids',
 'Infertility',
 'Fibromyalgia',
 "Alzheimer's Disease",
 'Bronchitis',
 'Antihistamines',
 'ECG',
 'Urinary Tract Infection',
 'Speech Therapy',
 'Joint Pain',
 'Occupational Therapy',
 'Brain Tumor',
 'Bronchodilators',
 'Multiple Sclerosis',
 'Homeopathy',
 'Cholesterol Medications',
 'Antipsychotics',
 'Leukemia',
 'Abdominal Pain',
 'Pregnancy',
 "Addison's Disease",
 'Allergies',
 'Asthma',
 'Psoriasis',
 'Blood Test',
 'Vaccinations',
 'Antibiotic Resistance',
 'Fever',
 'Sinusitis',
 'Nutritional Counseling',
 'Weight Management',
 'Thyroid Cancer',
 'Lung Cancer',
 'Chronic Obstructive Pulmonary Disease',
 'Lymphoma',
 'Cervical Cancer',
 'Sleep Hygiene',
 'Wound Care',
 'Allergy Shots',
 'Insulin',
 'Weight Loss',
 'Lupus',
 'Epilepsy',
 'Skin Cancer',
 'Liver Cancer',
 'Osteoporosis',
 'Diabetic Neuropathy',
 "Crohn's Disease",
 'Tuberculosis',
 'NSAIDs',
 'Antidepressants',
 'Gout',
 'Alternative Medicine',
 'Constipation',
 'CT Scan',
 'Endometriosis',
 'Gastritis',
 'Anemia',
 'Prostate Cancer',
 'Headache',
 'Food Allergies',
 'Cancer',
 'High Cholesterol',
 "Cushing's Syndrome",
 'Scoliosis',
 'Stress Management',
 'Polycystic Ovary Syndrome',
 'Corticosteroids',
 'Cough',
 'Exercise Regimen',
 'Rheumatoid Arthritis',
 'Back Pain',
 'Alcohol Moderation',
 'ACE Inhibitors',
 'Antibiotics',
 'Anxiety',
 'X-Ray',
 'Endoscopy',
 'HIV',
 'Herbal Remedies',
 'Smoking Cessation',
 'Asthma Attack',
 'Massage Therapy',
 'Pancreatic Cancer',
 'Immunotherapy',
 'Biopsy',
 'Antifungal Medications',
 'Hepatitis',
 'Morning Sickness',
 'Ulcer',
 'Swelling',
 'Statins',
 'Depression',
 'Inflammatory Bowel Disease',
 'Antiviral Medications',
 'Anxiolytics',
 'Hyperthyroidism',
 'Lactose Intolerance',
 'Beta Blockers',
 'Thyroid Nodules',
 'Premenstrual Syndrome',
 'Hypothyroidism',
 'Blood Pressure Medications',
 'Hydration',
 'Irritable Bowel Syndrome',
 'Chronic Fatigue Syndrome',
 'Metformin',
 'Diabetes',
 'Testicular Cancer',
 'Ulcerative Colitis',
 'Shortness of Breath',
 'High Blood Sugar',
 'Ovarian Cancer',
 'Colorectal Cancer',
 'Surgery',
 'Hormonal Imbalance',
 'Menstrual Cramps',
 'Diabetes Insipidus',
 'Anticoagulants',
 'Infection Control',
 'Hemorrhoids',
 "Parkinson's Disease",
 'Pneumonia',
 'Palpitations',
 'Melanoma',
 'Menopause',
 'Sun Protection',
 'Arthritis',
 'Ultrasound',
 'Insomnia',
 'Acupuncture',
 'Urinalysis',
 'Kidney Stones',
 'Fatigue',
 'Erectile Dysfunction',
 'Nausea',
 'Dizziness',
 'Rash',
 'Migraine',
 'Painkillers',
 'Asthma Inhaler',
 'Vomiting',
 'Physical Therapy',
 'MRI',
 'Antihypertensives']
entity_pool = list(set([e.replace('"','') for e in entity_pool]))

In [63]:
len(entity_pool)

170

In [64]:
# Generate English Questions with Multiple Entities
def generate_questions_multi_entity(allocation, entity_pool, variants_per_template=2):
    """
    allocation: dict, 模板类别 -> 生成问题数量
    entity_pool: list of str, 可用实体
    variants_per_template: 每个模板生成多少变体
    """
    questions = []

    for cat, num in allocation.items():
        templates = TEMPLATES[cat]
        for _ in range(num):
            tmpl = random.choice(templates)
            for _ in range(variants_per_template):
                # 从实体池随机选择实体
                entity_main = random.choice(entity_pool)
                entity_a = random.choice(entity_pool)
                entity_b = random.choice(entity_pool)
                process_task = random.choice(entity_pool)
                phenomenon = random.choice(entity_pool)
                claim = random.choice(entity_pool)
                
                # 填充模板槽位
                q = tmpl.replace("[ENTITY]", entity_main)
                q = q.replace("[ENTITY A]", entity_a)
                q = q.replace("[ENTITY B]", entity_b)
                q = q.replace("[PROCESS]", process_task)
                q = q.replace("[TASK]", process_task)
                q = q.replace("[PHENOMENON]", phenomenon)
                q = q.replace("[CLAIM]", claim)
                
                questions.append(q)
    return questions

# 多实体生成
# entity_pool = ["diabetes", "hypertension", "asthma", "cancer", "flu", "COVID-19", "migraine"]
questions_multi = generate_questions_multi_entity(allocation, entity_pool, variants_per_template=3)

# 查看前10个示例
print("Sample multi-entity questions:")
for q in questions_multi[:10]:
    print("-", q)

Sample multi-entity questions:
- Can you give an example of High Cholesterol?
- Can you give an example of Blood Test?
- Can you give an example of Antihistamines?
- Can you give an example of Asthma Inhaler?
- Can you give an example of Polycystic Ovary Syndrome?
- Can you give an example of Dizziness?
- What is Antihistamines?
- What is Biopsy?
- What is Diabetes?
- Can you give an example of Pancreatic Cancer?


In [65]:
questions_multi

['Can you give an example of High Cholesterol?',
 'Can you give an example of Blood Test?',
 'Can you give an example of Antihistamines?',
 'Can you give an example of Asthma Inhaler?',
 'Can you give an example of Polycystic Ovary Syndrome?',
 'Can you give an example of Dizziness?',
 'What is Antihistamines?',
 'What is Biopsy?',
 'What is Diabetes?',
 'Can you give an example of Pancreatic Cancer?',
 'Can you give an example of Homeopathy?',
 'Can you give an example of MRI?',
 'Can you give an example of CT Scan?',
 'Can you give an example of Constipation?',
 'Can you give an example of Insomnia?',
 'How does Anemia differ from Pain Management?',
 'How does Pancreatic Cancer differ from Testicular Cancer?',
 'How does Speech Therapy differ from Premenstrual Syndrome?',
 'What is Eczema?',
 'What is Lupus?',
 'What is Chronic Fatigue Syndrome?',
 'How does Anti-inflammatory Drugs differ from Food Allergies?',
 'How does CT Scan differ from Sleep Aids?',
 'How does Itching differ fr

In [66]:
len(questions_multi)

1500

## 5. Step 5 — Export results

In [67]:
questions_multi = random.sample(questions_multi, 500)

In [68]:
# 保存为 JSON Lines
output_file = "questions.jsonl"
with open(output_file, "w", encoding="utf-8") as f:
    for idx, q in enumerate(questions_multi, start=1):
        entry = {"_id": str(idx), "text": q, "metadata": {}}
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"{len(questions_multi)} questions saved to {output_file}")

500 questions saved to questions.jsonl
